# Gradient Descent from Scratch
---

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Training data: y = 2x
x_train = np.array([1, 2, 3, 4, 5])
y_train = np.array([2, 4, 6, 8, 10])

# Bad initial guesses
w = 10
b = 5

print("x_train:", x_train)
print("y_train:", y_train)
print(f"Initial w={w}, b={b}  <- bad starting values")


## 2. Cost Function
Measures how wrong our model is.

In [ ]:
def cost_function(x_train, y_train, w, b):
    m = x_train.shape[0]
    cost = 0
    for i in range(m):
        # prediction - actual, squared
        cost += (w * x_train[i] + b - y_train[i])**2
    return cost / (2 * m)

print("Cost with bad w=10, b=5:", "Cost =" , cost_function(x_train, y_train, w, b))
print("High cost = model is very wrong!")


## 3. Before Gradient Descent
See how bad our initial fit is.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: bad fit
x_line = np.linspace(0, 6, 100)
axes[0].scatter(x_train, y_train, color='blue', s=100, zorder=5, label='Actual data')
axes[0].plot(x_line, 10 * x_line + 5, color='red', linewidth=2, label='Bad fit: w=10, b=5')
axes[0].set_title('Before Gradient Descent')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: cost surface
w_vals = np.linspace(-5, 15, 100)
b_vals = np.linspace(-5, 15, 100)
W, B = np.meshgrid(w_vals, b_vals)
Z = np.array([[cost_function(x_train, y_train, wi, bi) for wi in w_vals] for bi in b_vals])

contour = axes[1].contourf(W, B, Z, levels=50, cmap='viridis')
axes[1].scatter([10], [5], color='red', s=200, zorder=5, label='Start: w=10, b=5')
axes[1].scatter([2], [0], color='lime', s=200, marker='*', zorder=5, label='Goal: w=2, b=0')
plt.colorbar(contour, ax=axes[1])
axes[1].set_title('Cost Surface J(w,b)')
axes[1].set_xlabel('w')
axes[1].set_ylabel('b')
axes[1].legend()

plt.tight_layout()
plt.show()


## 4. Compute Gradient
Tells us which direction to move w and b.

In [ ]:
def compute_gradient(x_train, y_train, w, b):
    m = x_train.shape[0]
    temp_w = 0
    temp_b = 0
    for i in range(m):
        # error = prediction - actual
        error = w * x_train[i] + b - y_train[i]
        temp_w += error * x_train[i]  # gradient for w
        temp_b += error               # gradient for b
    return temp_w / m, temp_b / m

dw, db = compute_gradient(x_train, y_train, w, b)
print(f"dw = {dw:.2f}  -> w is too large, needs to decrease")
print(f"db = {db:.2f}  -> b is too large, needs to decrease")


## 5. Gradient Descent
Repeatedly update w and b until cost is minimized.

In [ ]:
def gradient_descent(w, b, alpha=0.01, iterations=10000):
    cost_history = []
    w_history = []
    b_history = []

    for i in range(iterations):
        # compute gradient
        dw, db = compute_gradient(x_train, y_train, w, b)
        
        # update w and b
        w = w - alpha * dw
        b = b - alpha * db

        # save history
        cost_history.append(cost_function(x_train, y_train, w, b))
        w_history.append(w)
        b_history.append(b)

        # print every 1000 steps
        if i % 1000 == 0:
            print(f"Iteration {i:5d} | Cost: {cost_history[-1]:.6f} | w: {w:.4f} | b: {b:.6f}")

    return w, b, cost_history, w_history, b_history

# start from bad values
w, b = 10, 5
print("Starting...")
w_final, b_final, cost_history, w_history, b_history = gradient_descent(w, b)
print(f"Final w = {w_final:.6f}  (expected 2)")
print(f"Final b = {b_final:.2e}   (expected 0)")


## 6. Cost Dropping Over Time
Left = first 100 steps (fast). Right = rest (slow fine-tuning).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(cost_history[:100], color='crimson', linewidth=2)
axes[0].set_title('Cost - First 100 Iterations (fast drop)')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Cost')
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1000, 10000), cost_history[1000:], color='steelblue', linewidth=2)
axes[1].set_title('Cost - Iterations 1000 to 10000 (slow fine-tuning)')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Cost')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 7. Path on Cost Surface
White line = the journey of w and b toward the goal.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

w_vals = np.linspace(-1, 12, 200)
b_vals = np.linspace(-1, 8, 200)
W, B = np.meshgrid(w_vals, b_vals)
Z = np.array([[cost_function(x_train, y_train, wi, bi) for wi in w_vals] for bi in b_vals])

contour = ax.contourf(W, B, Z, levels=40, cmap='coolwarm', alpha=0.8)
plt.colorbar(contour, ax=ax, label='Cost J(w,b)')

# plot path every 50 steps
step = 50
ax.plot(w_history[::step], b_history[::step], 'w-o', markersize=4, linewidth=1.5, label='GD path')
ax.scatter([w_history[0]], [b_history[0]], color='red', s=200, zorder=10, label='Start (w=10,b=5)')
ax.scatter([w_final], [b_final], color='lime', s=300, marker='*', zorder=10, label='End (w=2,b=0)')

ax.set_title('Gradient Descent Path on Cost Surface')
ax.set_xlabel('w')
ax.set_ylabel('b')
ax.legend()
plt.tight_layout()
plt.show()


## 8. Before vs After

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x_line = np.linspace(0, 7, 100)

axes[0].scatter(x_train, y_train, color='blue', s=100, zorder=5, label='Actual data')
axes[0].plot(x_line, 10 * x_line + 5, color='red', linewidth=2, label='Before: w=10, b=5')
axes[0].set_title('Before Gradient Descent')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].scatter(x_train, y_train, color='blue', s=100, zorder=5, label='Actual data')
axes[1].plot(x_line, w_final * x_line + b_final, color='green', linewidth=2,
             label=f'After: w={w_final:.2f}, b={b_final:.1e}')
axes[1].set_title('After Gradient Descent')
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 9. Predictions

In [ ]:
print("Predictions using learned model:")
print("-" * 35)
for x_new in [6, 7, 10]:
    pred = w_final * x_new + b_final
    expected = 2 * x_new
    print(f"x={x_new} -> predicted: {pred:.4f}  (expected: {expected})")
